# Lab 20-03 · Bandgap References over Temperature, Process and Trim

**Predict first**, then run. The handout is [README.md](README.md). The PNP is the chapter's illustrative T130 device.

In [1]:
import math
import collections
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import brentq
import bandgap as b

p = b.PNP()
T0 = 300.0
temps = np.arange(-40, 125.001, 0.5) + 273.15


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {m.group(1): float(m.group(2)) for m in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · The two slopes

*Predict* V_BE at -40 C and 125 C for 5 uA with constant and PTAT bias.

In [2]:
i0, n = 5e-6, 8.0
for m in (0, 1):
    vals = [p.vbe(i0 * (t / T0) ** m, t) for t in (233.15, 398.15)]
    print(f"m = {m}: V_BE(-40 C) = {vals[0]:.4f} V, V_BE(125 C) = {vals[1]:.4f} V")
vbe = np.array([p.vbe(i0 * t / T0, t) for t in temps])
k = b.k_zero_tc(p.vbe(i0, T0), n, p.vg0, p.eta, 1, T0)
ptat = k * np.array([b.vt(t) for t in temps]) * math.log(n)
plt.plot(temps - 273.15, vbe, label="V_BE (PTAT bias)"); plt.plot(temps - 273.15, ptat, label=f"K V_T ln N, K = {k:.2f}")
plt.plot(temps - 273.15, vbe + ptat, label="sum"); plt.xlabel("T (C)"); plt.ylabel("V"); plt.legend(); plt.show()

m = 0: V_BE(-40 C) = 0.8101 V, V_BE(125 C) = 0.4784 V
m = 1: V_BE(-40 C) = 0.8050 V, V_BE(125 C) = 0.4881 V


## Part 2 · The bandgap in ngspice

*Predict* the output at 300 K, the peak temperature and the box TC of Example 20-03.2.

In [3]:
tc = brentq(lambda x: b.vref_closed_form(233.15, p.vg0, p.eta, 1, x) - b.vref_closed_form(398.15, p.vg0, p.eta, 1, x), 250, 380)
r1 = b.vt(T0) * math.log(n) / i0
r2 = b.k_zero_tc(p.vbe(i0, T0), n, p.vg0, p.eta, 1, tc) * r1
vals = [b.opamp_bandgap(t, r1, r2)["vref"] for t in temps]
vn = b.opamp_bandgap(T0, r1, r2)["vref"]
print(f"T_c = {tc:.2f} K, R1 = {r1 / 1e3:.3f} kohm, R2 = {r2 / 1e3:.2f} kohm, V(300 K) = {vn:.5f} V, box TC = {b.box_tc(temps, vals, vn):.2f} ppm/K")
print(ngspice("bandgap.cir"))
for t_c in (300.15, tc, 333.15):
    plt.plot(temps - 273.15, [b.vref_closed_form(t, p.vg0, p.eta, 1, t_c) for t in temps], label=f"zero TC at {t_c - 273.15:.0f} C")
plt.xlabel("T (C)"); plt.ylabel("V_REF (V)"); plt.legend(); plt.show()

T_c = 312.00 K, R1 = 10.752 kohm, R2 = 115.23 kohm, V(300 K) = 1.25463 V, box TC = 9.10 ppm/K
{'vnom': 1.254634, 'vmax': 1.254675, 'vmin': 1.252791, 'tlo': 25.5789, 'thi': 52.35688, 'vnomb': 1.241872, 'tpk': 38.96789}


## Part 3 · Trim Monte Carlo

*Predict* whether a one-temperature trim improves the median box TC when parts have both I_S spread and op-amp offset.

In [4]:
rng = np.random.default_rng(2003)
z = rng.standard_normal((400, 2))
grid = np.arange(-40, 125.0001, 5.0) + 273.15


def part(z1, z2, sigma_os=0.5e-3):
    pp = b.PNP(is0=p.is0 * math.exp(0.1 * z1))
    vos = sigma_os * z2
    f = lambda t, r2x=r2: b.opamp_bandgap(t, r1, r2x, pnp=pp, vos=vos)["vref"]
    untrim = b.box_tc(grid, [f(t) for t in grid], vn)
    r2t = brentq(lambda r: f(T0, r) - vn, 0.8 * r2, 1.25 * r2, xtol=1e-9)
    return f(T0), untrim, b.box_tc(grid, [f(t, r2t) for t in grid], vn)


res = np.array([part(z1, z2) for z1, z2 in z])
print(f"sigma of V(300 K) = {res[:, 0].std(ddof=1) * 1e3:.2f} mV")
print(f"untrimmed box TC: median {np.median(res[:, 1]):.2f}, 95th percentile {np.percentile(res[:, 1], 95):.2f} ppm/K")
print(f"trimmed box TC:   median {np.median(res[:, 2]):.2f}, 95th percentile {np.percentile(res[:, 2], 95):.2f} ppm/K, "
      f"{np.mean(res[:, 2] < 15) * 100:.2f} % within 15 ppm/K")
res0 = np.array([part(z1, z2, sigma_os=0.0) for z1, z2 in z[:40]])
print(f"offset-free parts after trim: box TC from {res0[:, 2].min():.4f} to {res0[:, 2].max():.4f} ppm/K")
plt.hist(res[:, 1], bins=30, alpha=0.6, label="untrimmed"); plt.hist(res[:, 2], bins=30, alpha=0.6, label="trimmed at 300 K")
plt.xlabel("box TC (ppm/K)"); plt.ylabel("parts"); plt.legend(); plt.show()

sigma of V(300 K) = 7.06 mV
untrimmed box TC: median 11.37, 95th percentile 18.14 ppm/K
trimmed box TC:   median 15.39, 95th percentile 35.15 ppm/K, 47.75 % within 15 ppm/K
offset-free parts after trim: box TC from 9.0969 to 9.0969 ppm/K


## Part 4 · A 0.6 V current-mode reference

*Predict* its output at -40 C and 125 C (Example 20-03.4).

In [5]:
res4 = b.current_mode_bandgap(T0, r1, r2, 1.0)
r3 = 0.6 / res4["i_branch"]
for t in (233.15, 300.0, 398.15):
    print(f"T = {t - 273.15:6.1f} C: V_REF = {b.current_mode_bandgap(t, r1, r2, r3)['vref']:.5f} V")
print(ngspice("banba.cir"))

T =  -40.0 C: V_REF = 0.59912 V
T =   26.9 C: V_REF = 0.60000 V
T =  125.0 C: V_REF = 0.59912 V
{'vnom': 0.6000006, 'vmax': 0.60002, 'vmin': 0.5991189}


## Part 5 · Curvature compensation against eta

*Predict* how the box TC of a first-order and a compensated (alpha = 2) design change as the true eta moves from 2.6 to 3.4.

In [6]:
def box_c(c):
    if abs(c) < 1e-12:
        return 0.0
    f = lambda t, x: p.vg0 + c * b.vt(t) * (1 - math.log(t / x))
    x = brentq(lambda y: f(233.15, y) - f(398.15, y), 200, 420)
    vals = [f(t, x) for t in temps]
    return b.box_tc(temps, vals, f(T0, x))


etas = np.linspace(2.6, 3.4, 33)
plt.plot(etas, [box_c(e - 1) for e in etas], label="first order")
plt.plot(etas, [box_c(e - 3) for e in etas], label="compensated, alpha = 2")
plt.xlabel("true eta"); plt.ylabel("box TC, -40 to 125 C (ppm/K)"); plt.legend(); plt.show()

## Part 6 · The beta multiplier's two operating points

*Predict* where M1's curve and the M2 + R branch's curve cross.

In [7]:
nm = b.NMOS()
vg = np.linspace(0.15, 0.45, 61)
i1 = [nm.i_d(x, 0.0, 10) for x in vg]
i2 = [brentq(lambda s: nm.i_d(x, s, 40) - s / 100e3, 0.0, x, xtol=1e-16) / 100e3 for x in vg]
iref = b.beta_multiplier(100e3, 4, 10)
print(f"operating point: I = {iref * 1e6:.3f} uA")
plt.semilogy(vg, i1, label="M1 (W/L = 10)"); plt.semilogy(vg, i2, label="M2 (W/L = 40) with R = 100 kohm")
plt.axhline(iref, color="k", lw=0.8); plt.xlabel("gate voltage (V)"); plt.ylabel("current (A)"); plt.legend(); plt.show()

operating point: I = 0.372 uA
